# 🎙️ Whisper-Small Fine-Tuning for Uzbek Speech-to-Text

Ushbu notebook `openai/whisper-small` modelini `islomov/news_youtube_uzbek_speech_dataset` datasetining 1 soatlik audiosi asosida o'zbek tiliga fine-tune qilish va modelni to'g'ridan-to'g'ri Hugging Face Hub'ga yuklash uchun mo'ljallangan.

### ⚡ Talablar va tayyorgarlik:
1. **GPU faollashtirish:** Menyu orqali `Runtime` -> `Change runtime type` -> `T4 GPU` tanlang.
2. **Hugging Face Token:** Chap menyudan 🔑 (Secrets) tugmasini bosing, `HF_TOKEN` nomli yangi secret yaratib, HF tokeningizni qo'ying va Notebook access ni yoqing.

In [ ]:
# 1. GPU tekshiruvi
!nvidia-smi

In [ ]:
# 2. Kerakli kutubxonalarni o'rnatish
!pip install -q --upgrade pip
!pip install -q transformers datasets[audio] evaluate jiwer accelerate soundfile librosa huggingface_hub

In [ ]:
# 3. Hugging Face tizimiga kirish
import os
from huggingface_hub import login, whoami

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✅ Hugging Face tizimiga Colab Secrets orqali muvaffaqiyatli kirildi!')
except Exception as e:
    print('Colab Secrets orqali token topilmadi, interaktiv login qilinadi:')
    from huggingface_hub import notebook_login
    notebook_login()

user_info = whoami()
hf_username = user_info['name']
print(f'Hugging Face Username: {hf_username}')

In [ ]:
# 4. Asosiy parametrlar va sozlamalar
MODEL_ID = "openai/whisper-small"
LANGUAGE = "uzbek"
TASK = "transcribe"
TARGET_DURATION_SECONDS = 3600.0  # 1 soat (3600 soniya)
HUB_MODEL_ID = f"{hf_username}/whisper-small-uzbek-1h"

print(f"Target HF Model ID: {HUB_MODEL_ID}")

In [ ]:
# 5. Datasetni yuklash va 1 soatlik audioni ajratish
from datasets import load_dataset, Audio

print("Dataset yuklanmoqda...")
raw_dataset = load_dataset("islomov/news_youtube_uzbek_speech_dataset", split="train")
raw_dataset = raw_dataset.cast_column("audio", Audio(sampling_rate=16000))

selected_indices = []
total_sec = 0.0

for idx, sample in enumerate(raw_dataset):
    dur = len(sample["audio"]["array"]) / 16000.0
    if total_sec + dur > TARGET_DURATION_SECONDS:
        break
    total_sec += dur
    selected_indices.append(idx)

subset = raw_dataset.select(selected_indices)
print(f"Jami tanlangan: {len(subset)} ta audio namuna | {total_sec / 60:.2f} daqiqa ({total_sec:.1f} soniya)")

split_dataset = subset.train_test_split(test_size=0.2, seed=42)
print(f"Train: {len(split_dataset['train'])} ta | Test: {len(split_dataset['test'])} ta")

In [ ]:
# 6. Whisper Feature Extractor, Tokenizer va Processor
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_ID)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = tokenizer(batch["text"]).input_ids
    return batch

print("Mel-spectrogram va tokenizatsiya qilinmoqda...")
prepared_dataset = split_dataset.map(
    prepare_dataset,
    remove_columns=split_dataset["train"].column_names,
    num_proc=2
)
print("Tayyorgarlik muvaffaqiyatli yakunlandi!")

In [ ]:
# 7. Dynamic Data Collator with Padding
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        bos_id = self.processor.tokenizer.bos_token_id
        if bos_id is not None and labels.shape[1] > 0 and (labels[:, 0] == bos_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
# 8. Baholash metrikalari (WER va CER)
import numpy as np
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = np.where(pred.label_ids != -100, pred.label_ids, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    valid_pairs = [(p, r) for p, r in zip(pred_str, label_str) if len(r.strip()) > 0]
    if not valid_pairs:
        return {"wer": 0.0, "cer": 0.0}

    filtered_preds, filtered_refs = zip(*valid_pairs)
    wer = 100.0 * wer_metric.compute(predictions=list(filtered_preds), references=list(filtered_refs))
    cer = 100.0 * cer_metric.compute(predictions=list(filtered_preds), references=list(filtered_refs))
    return {"wer": wer, "cer": cer}

In [ ]:
# 9. Modelni yuklash va Seq2SeqTrainer sozlamalari
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False  # gradient checkpointing uchun zarur

# GenerationConfig sozlamalari (O'zbek tilini birlamchi qilib belgilash)
if hasattr(model, "generation_config") and model.generation_config is not None:
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.generation_config.forced_decoder_ids = None
    model.generation_config.suppress_tokens = []

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-uzbek-1h",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=50,
    max_steps=600,
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=150,
    eval_steps=150,
    logging_steps=25,
    report_to=["tensorboard"],
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
    hub_model_id=HUB_MODEL_ID
)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

In [ ]:
# 10. Fine-tuning jarayonini boshlash
print("Training boshlandi...")
trainer.train()

In [ ]:
# 11. Model va Processorni Hugging Face Hub'ga push qilish
kwargs = {
    "dataset_tags": "islomov/news_youtube_uzbek_speech_dataset",
    "dataset": "Uzbek Speech Dataset (1 Hour)",
    "language": "uz",
    "model_name": "Whisper Small Uzbek (1h)",
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

trainer.push_to_hub(**kwargs)
processor.push_to_hub(HUB_MODEL_ID)
print(f"🎉 Model muvaffaqiyatli yuklandi: https://huggingface.co/{HUB_MODEL_ID}")

In [ ]:
# 12. Modelni test qilish (Inference demo)
from transformers import pipeline

pipe = pipeline("automatic-speech-recognition", model=HUB_MODEL_ID)
sample = split_dataset["test"][0]

result = pipe(sample["audio"]["array"])
print("--------------------------------------------------")
print("ASL MATN (Reference):    ", sample["text"])
print("WHISPER (Prediction):    ", result["text"])
print("--------------------------------------------------")